# 🎓 LectureScribe — Kaggle Edition

Transforms lecture videos (YouTube / direct URL) into structured, exam-ready PDF notes.

**Pipeline:** YouTube URL → yt-dlp → Whisper large-v3 (T4 GPU) → NVIDIA Nemotron-Ultra → Mermaid diagrams → ReportLab PDF

---
### Before running:
1. Add your NVIDIA API key to **Kaggle Secrets** with the name `NVIDIA_API_KEY`
2. Enable **Internet** in notebook settings (right panel)
3. Enable **GPU T4 x2** accelerator
4. Fill in the **CONFIG cell** (Cell 3) and run all cells top to bottom

In [ ]:
# ── CELL 1: Environment Setup ─────────────────────────────────────────────────
# Installs all dependencies. Run once per session (~3-4 minutes).

import subprocess, sys

print('📦 Installing Python packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'openai-whisper',
    'yt-dlp',
    'openai',
    'python-dotenv',
    'reportlab',
    'rich',
    'Pillow',
    'ffmpeg-python'
], check=True)

print('📦 Installing Node.js Mermaid CLI...')
subprocess.run(['npm', 'install', '-g', '@mermaid-js/mermaid-cli'], 
               capture_output=True)

# Verify mmdc
r = subprocess.run(['mmdc', '--version'], capture_output=True, text=True)
if r.returncode == 0:
    print(f'✅ mmdc {r.stdout.strip()} ready')
else:
    print('⚠️  mmdc not found in PATH — diagrams will use placeholders')

# Verify ffmpeg
r2 = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('✅ ffmpeg ready' if r2.returncode == 0 else '❌ ffmpeg missing')

print('\n✅ Setup complete.')

In [ ]:
# ── CELL 2: Clone LectureScribe from GitHub ───────────────────────────────────

import os, sys

REPO_URL = 'https://github.com/vansh-kumar-007/lecturescribe.git'
REPO_DIR = '/kaggle/working/lecturescribe'

if os.path.exists(REPO_DIR):
    print('Repo already cloned. Pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

# Add repo to Python path so we can import modules directly
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.makedirs(f'{REPO_DIR}/outputs/diagrams', exist_ok=True)
os.makedirs(f'{REPO_DIR}/outputs/fonts', exist_ok=True)

print(f'✅ LectureScribe ready at {REPO_DIR}')

In [ ]:
# ── CELL 3: CONFIG — Fill this in before running ──────────────────────────────

# Source type: 'youtube' or 'url'
SOURCE_TYPE = 'youtube'

# Paste your video URL here
VIDEO_URL = 'https://www.youtube.com/watch?v=XXXXXXXXXXX'

# Optional: override the lecture title in the PDF (leave '' to use Nemotron's title)
TITLE_OVERRIDE = ''

# Whisper model: 'large-v3' (best, fits on T4), 'medium' (faster, less accurate)
WHISPER_MODEL = 'large-v3'

# Language hint for Whisper. 'auto' for auto-detect.
# For Hinglish (Hindi+English mixed), use 'hi' for better results.
WHISPER_LANGUAGE = 'auto'  # 'auto' | 'en' | 'hi'

# Output directory
OUTPUT_DIR = f'{REPO_DIR}/outputs'

print('✅ Config set.')
print(f'   Source : {SOURCE_TYPE}')
print(f'   URL    : {VIDEO_URL}')
print(f'   Model  : Whisper {WHISPER_MODEL}')
print(f'   Lang   : {WHISPER_LANGUAGE}')

In [ ]:
# ── CELL 4: Load NVIDIA API Key from Kaggle Secrets ──────────────────────────
# Add your key in: Notebook Settings → Secrets → Add Secret
# Name it exactly: NVIDIA_API_KEY

from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
NVIDIA_API_KEY = secrets.get_secret('NVIDIA_API_KEY')

# Write to .env so utils.get_env() picks it up
with open(f'{REPO_DIR}/.env', 'w') as f:
    f.write(f'NVIDIA_API_KEY={NVIDIA_API_KEY}\n')

print('✅ NVIDIA API key loaded from Kaggle Secrets.')

In [ ]:
# ── CELL 5: Step 1 — Download & Extract Audio ────────────────────────────────

import os, time
os.chdir(REPO_DIR)  # run from repo root so relative paths work

from utils import extract_audio

print('🎵 Extracting audio...')
t0 = time.time()
audio_path = extract_audio(VIDEO_URL, output_dir=OUTPUT_DIR)
print(f'✅ Audio extracted in {time.time()-t0:.1f}s → {audio_path}')

In [ ]:
# ── CELL 6: Step 2 — Transcribe with Whisper ─────────────────────────────────
# T4 GPU: ~1.5-2 hrs for a 12-hour video (vs ~6 hrs on RTX 3050)

import time, warnings
import torch
import whisper

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🎙 Transcribing with Whisper {WHISPER_MODEL} on {device.upper()}...')

if device == 'cpu':
    print('⚠️  No GPU detected! Make sure GPU is enabled in notebook settings.')

t0 = time.time()

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    model = whisper.load_model(WHISPER_MODEL, device=device)

lang = None if WHISPER_LANGUAGE == 'auto' else WHISPER_LANGUAGE

result = model.transcribe(
    audio_path,
    fp16=(device == 'cuda'),
    language=lang,
    verbose=False
)

transcript = result['text'].strip()
word_count = len(transcript.split())
elapsed = time.time() - t0

# Save transcript
transcript_path = f'{OUTPUT_DIR}/transcript.txt'
with open(transcript_path, 'w', encoding='utf-8') as f:
    f.write(transcript)

print(f'✅ Transcription complete in {elapsed/60:.1f} min')
print(f'   Words    : ~{word_count}')
print(f'   Language : {result.get("language", "unknown")}')
print(f'   Saved to : {transcript_path}')
print(f'\nPreview (first 500 chars):')
print(transcript[:500])

In [ ]:
# ── CELL 7: (Optional) Load existing transcript instead of re-transcribing ────
# Run this cell INSTEAD of Cell 6 if you already have a transcript saved.

transcript_path = f'{OUTPUT_DIR}/transcript.txt'

if os.path.exists(transcript_path):
    with open(transcript_path, 'r', encoding='utf-8') as f:
        transcript = f.read()
    print(f'✅ Loaded existing transcript: ~{len(transcript.split())} words')
else:
    print('❌ No transcript found. Run Cell 6 first.')

In [ ]:
# ── CELL 8: Step 3 — Analyze with NVIDIA Nemotron ───────────────────────────
# Chunks transcript and sends to Nemotron-Ultra.
# Includes 15s pauses + auto-retry on rate limits.

import json, time
from nemotron import analyze_transcript

print('🧠 Analyzing transcript with Nemotron-Ultra...')
t0 = time.time()

notes = analyze_transcript(transcript)

if TITLE_OVERRIDE:
    notes['lecture_title'] = TITLE_OVERRIDE

# Save JSON for debugging / re-use
notes_path = f'{OUTPUT_DIR}/notes.json'
with open(notes_path, 'w', encoding='utf-8') as f:
    json.dump(notes, f, indent=2)

block_types = [b['type'] for b in notes.get('blocks', [])]
from collections import Counter
counts = Counter(block_types)

print(f'✅ Analysis complete in {(time.time()-t0)/60:.1f} min')
print(f'   Title   : {notes.get("lecture_title")}')
print(f'   Subject : {notes.get("subject")}')
print(f'   Blocks  : {len(block_types)} total')
for btype, count in counts.items():
    print(f'     {btype:12s}: {count}')
print(f'   Saved to: {notes_path}')

In [ ]:
# ── CELL 9: (Optional) Load existing notes.json instead of re-running Nemotron
# Run this INSTEAD of Cell 8 if notes.json already exists.

import json
notes_path = f'{OUTPUT_DIR}/notes.json'

if os.path.exists(notes_path):
    with open(notes_path, 'r', encoding='utf-8') as f:
        notes = json.load(f)
    print(f'✅ Loaded existing notes: {len(notes.get("blocks", []))} blocks')
    print(f'   Title: {notes.get("lecture_title")}')
else:
    print('❌ No notes.json found. Run Cell 8 first.')

In [ ]:
# ── CELL 10: Step 4 — Render Diagrams ────────────────────────────────────────

import time
from diagram_renderer import render_diagrams

print('📊 Rendering Mermaid diagrams...')
t0 = time.time()

diagram_paths = render_diagrams(
    notes['blocks'],
    OUTPUT_DIR,
    notes.get('lecture_title', 'lecture')
)

print(f'✅ Diagrams rendered in {time.time()-t0:.1f}s')
print(f'   {len(diagram_paths)} diagram(s) rendered')
for idx, path in diagram_paths.items():
    print(f'   Block {idx}: {os.path.basename(path)}')

In [ ]:
# ── CELL 11: Step 5 — Generate PDF ───────────────────────────────────────────

import time
from pdf_renderer import render_pdf

print('📄 Generating PDF...')
t0 = time.time()

pdf_path = render_pdf(notes, diagram_paths, OUTPUT_DIR)

print(f'✅ PDF generated in {time.time()-t0:.1f}s')
print(f'   Saved to: {pdf_path}')

# Show file size
size_kb = os.path.getsize(pdf_path) / 1024
print(f'   Size    : {size_kb:.1f} KB')

In [ ]:
# ── CELL 12: Download PDF ─────────────────────────────────────────────────────
# Copy PDF to /kaggle/working/ root for easy download from the Output panel.

import shutil, os
from pathlib import Path

pdf_name = Path(pdf_path).name
dest = f'/kaggle/working/{pdf_name}'
shutil.copy2(pdf_path, dest)

print(f'✅ PDF ready for download!')
print(f'   Go to the Output panel (right side) → find "{pdf_name}" → Download')
print(f'   Direct path: {dest}')

# Also copy transcript and notes.json for reference
shutil.copy2(f'{OUTPUT_DIR}/transcript.txt', '/kaggle/working/transcript.txt')
shutil.copy2(f'{OUTPUT_DIR}/notes.json', '/kaggle/working/notes.json')
print('   transcript.txt and notes.json also available in Output panel.')

---
## 💡 Tips

**Hinglish videos:** Set `WHISPER_LANGUAGE = 'hi'` in Cell 3 for better accuracy on Hindi+English mixed content.

**Long videos (6-12 hrs):** Whisper on T4 takes ~1.5-2 hrs. Start the notebook, let it run, come back. Kaggle sessions stay alive for up to 12 hours.

**Rate limits:** Nemotron has a ~32 request/session limit. LectureScribe handles this automatically with 60s retry waits. A 12-hour video may produce 15-20 chunks — expect ~10-20 min for Nemotron analysis after retries.

**Re-running without re-transcribing:** After Cell 6 runs once, use Cell 7 to load the saved transcript. Similarly, use Cell 9 to reload notes.json without calling Nemotron again.

**Local Udemy courses:** Run LectureScribe locally on your machine using SRT mode — it's instant and doesn't need GPU at all.